**Steps**

1. Load the CheXpert disease labels and keep only labels with over 10% prevalence.
2. Keep only studies in the official training split.
3. Sample 2,200 studies (tried 2 methods that failed, then plain random sampling worked).
4. Load MIMIC-IV admissions and match each study to a hospital admission.
5. Check how many of the 2,200 sampled studies actually linked to MIMIC-IV.
6. Fix the order: filter to linkable studies first, then sample 2,200 again.
7. Check the final disease balance and patient count.
8. Build image and text reference tables for the same 2,200 studies.
9. Save all four output files.

**Output:** `1_structured_reference.csv`, `matched_final.csv`, `2_image_reference.csv`, `3_text_reference.csv`

In [9]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

base_path = '/content/drive/MyDrive/dissertation_project/data'
raw_path = f'{base_path}/raw'
processed_path = f'{base_path}/processed'
os.makedirs(processed_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# PhysioNet login
import getpass
physionet_user = input("Enter your PhysioNet username: ") #guntreddibhavishya@gmail.com
physionet_pass = getpass.getpass("Enter your PhysioNet password: ") #######

Enter your PhysioNet username: Bhavishyaguntreddi
Enter your PhysioNet password: ··········


In [11]:
# Load CheXpert disease labels, keep only the 7 labels over 10% prevalence
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv.gz

chexpert_labels = pd.read_csv(f'{raw_path}/mimic-cxr-2.0.0-chexpert.csv.gz')
print("CheXpert labels loaded:", chexpert_labels.shape)

--2026-08-19 20:40:12--  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/mimic-cxr-2.0.0-chexpert.csv.gz’ not modified on server. Omitting download.

CheXpert labels loaded: (227827, 16)


In [12]:
# Check disease prevalence across all studies
disease_columns = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                    'Lung Opacity', 'No Finding', 'Pleural Effusion',
                    'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices']

total_studies = len(chexpert_labels)
for disease in disease_columns:
    positive_count = (chexpert_labels[disease] == 1.0).sum()
    prevalence_pct = (positive_count / total_studies) * 100
    print(f"{disease:<30} {positive_count:>10} {prevalence_pct:>8.2f}%")

Atelectasis                         45808    20.11%
Cardiomegaly                        44845    19.68%
Consolidation                       10778     4.73%
Edema                               27018    11.86%
Enlarged Cardiomediastinum           7179     3.15%
Fracture                             4390     1.93%
Lung Lesion                          6284     2.76%
Lung Opacity                        51525    22.62%
No Finding                          75455    33.12%
Pleural Effusion                    54300    23.83%
Pleural Other                        2011     0.88%
Pneumonia                           16556     7.27%
Pneumothorax                        10358     4.55%
Support Devices                     66558    29.21%


In [13]:
# Keep only the 7 diseases over 10% prevalence, drop studies with none of them
final_diseases = ['No Finding', 'Support Devices', 'Pleural Effusion',
                   'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']

chexpert_filtered = chexpert_labels[['subject_id', 'study_id'] + final_diseases]

no_relevant_label = chexpert_filtered[final_diseases].isna().all(axis=1)
chexpert_final = chexpert_filtered[~no_relevant_label].copy()
print("Studies with a usable label among the 7 diseases:", chexpert_final.shape)

Studies with a usable label among the 7 diseases: (215955, 9)


In [14]:
# Keep only studies in the official training split
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-split.csv.gz

split_data = pd.read_csv(f'{raw_path}/mimic-cxr-2.0.0-split.csv.gz')
train_study_ids = split_data[split_data['split'] == 'train']['study_id'].unique()

train_labelled = chexpert_final[chexpert_final['study_id'].isin(train_study_ids)]
print("Studies in the official train split with a usable label:", train_labelled.shape)

--2026-08-19 20:40:14--  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-split.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/mimic-cxr-2.0.0-split.csv.gz’ not modified on server. Omitting download.

Studies in the official train split with a usable label: (211135, 9)


In [15]:
# Attempt 3 (kept): plain random sample of 2,200 studies
target_size = 2200
sampled_df = train_labelled.sample(n=target_size, random_state=42)

print("Sampled pool shape:", sampled_df.shape)
for disease in final_diseases:
    pct = (sampled_df[disease] == 1.0).sum() / len(sampled_df) * 100
    print(f"{disease:<20} {pct:>6.2f}%")

Sampled pool shape: (2200, 9)
No Finding            34.77%
Support Devices       32.32%
Pleural Effusion      25.55%
Lung Opacity          23.68%
Atelectasis           22.45%
Cardiomegaly          21.45%
Edema                 13.14%


In [16]:
# Load MIMIC-IV admissions and CXR metadata, match study time to admission time
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz

!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz

admissions = pd.read_csv(f'{raw_path}/admissions.csv.gz')
metadata = pd.read_csv(f'{raw_path}/mimic-cxr-2.0.0-metadata.csv.gz')

metadata['StudyDateTime'] = pd.to_datetime(
    metadata['StudyDate'].astype(str) + metadata['StudyTime'].astype(int).astype(str).str.zfill(6),
    format='%Y%m%d%H%M%S'
)

admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

print("Admissions loaded:", admissions.shape)
print("Metadata loaded:", metadata.shape)

--2026-08-19 20:40:17--  https://physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/admissions.csv.gz’ not modified on server. Omitting download.

--2026-08-19 20:40:17--  https://physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awa

In [17]:
# Match each study to exactly one hospital admission (12hr buffer before admittime)
merged = metadata.merge(admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime']],
                          on='subject_id', how='inner')

window_before = pd.Timedelta(hours=12)
matched = merged[
    (merged['StudyDateTime'] >= merged['admittime'] - window_before) &
    (merged['StudyDateTime'] <= merged['dischtime'])
]

matched_unique = matched.drop_duplicates(subset=['study_id', 'hadm_id'])
matched_unique['time_gap'] = (matched_unique['StudyDateTime'] - matched_unique['admittime']).abs()
matched_final = matched_unique.sort_values('time_gap').drop_duplicates(subset=['study_id'], keep='first')

print("Studies with exactly one confirmed hospital admission:", matched_final.shape)

Studies with exactly one confirmed hospital admission: (164287, 17)


/tmp/ipykernel_595/1628495267.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_unique['time_gap'] = (matched_unique['StudyDateTime'] - matched_unique['admittime']).abs()


In [18]:
# Check how many of the 2,200 sampled studies actually linked to MIMIC-IV
sampled_study_ids = sampled_df['study_id'].unique()
linked_sampled = matched_final[matched_final['study_id'].isin(sampled_study_ids)]
print("Of the 2,200 sampled studies, how many have a MIMIC-IV link:", linked_sampled['study_id'].nunique())

Of the 2,200 sampled studies, how many have a MIMIC-IV link: 1629


In [19]:
#Fix: filter to linkable studies first, then sample 2,200 again
linkable_pool = train_labelled[train_labelled['study_id'].isin(matched_final['study_id'])]
print("Total linkable studies available:", linkable_pool.shape)

final_sampled_df = linkable_pool.sample(n=2200, random_state=42)
print("Final sampled pool (all guaranteed to link to MIMIC-IV):", final_sampled_df.shape)

Total linkable studies available: (153767, 9)
Final sampled pool (all guaranteed to link to MIMIC-IV): (2200, 9)


In [20]:
# Check disease balance after re-sampling
for disease in final_diseases:
    pct = (final_sampled_df[disease] == 1.0).sum() / len(final_sampled_df) * 100
    print(f"{disease:<20} {pct:>6.2f}%")

print("\nUnique patients in final training pool:", final_sampled_df['subject_id'].nunique())

No Finding            25.95%
Support Devices       37.55%
Pleural Effusion      29.14%
Lung Opacity          27.59%
Atelectasis           25.14%
Cardiomegaly          25.36%
Edema                 14.41%

Unique patients in final training pool: 2014


In [21]:
# Load image and report file lists
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimic-cxr/2.1.0/cxr-record-list.csv.gz

!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz

cxr_records = pd.read_csv(f'{raw_path}/cxr-record-list.csv.gz')
cxr_studies = pd.read_csv(f'{raw_path}/cxr-study-list.csv.gz')

print("cxr_records:", cxr_records.shape)
print("cxr_studies:", cxr_studies.shape)

--2026-08-19 20:40:28--  https://physionet.org/files/mimic-cxr/2.1.0/cxr-record-list.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/cxr-record-list.csv.gz’ not modified on server. Omitting download.

--2026-08-19 20:40:29--  https://physionet.org/files/mimic-cxr/2.1.0/cxr-study-list.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting

In [22]:
# Build image and text reference tables for the final 2,200 studies
image_ref = cxr_records[cxr_records['study_id'].isin(final_sampled_df['study_id'])][
    ['subject_id', 'study_id', 'dicom_id', 'path']
].rename(columns={'path': 'image_path'})

text_ref = cxr_studies[cxr_studies['study_id'].isin(final_sampled_df['study_id'])][
    ['subject_id', 'study_id', 'path']
].rename(columns={'path': 'report_path'})

print("Image reference:", image_ref.shape)
print("Text reference:", text_ref.shape)

Image reference: (3340, 4)
Text reference: (2200, 3)


In [23]:
# Save all four output files
final_sampled_df.to_csv(f'{processed_path}/1_structured_reference.csv', index=False)
matched_final.to_csv(f'{processed_path}/matched_final.csv', index=False)
image_ref.to_csv(f'{processed_path}/2_image_reference.csv', index=False)
text_ref.to_csv(f'{processed_path}/3_text_reference.csv', index=False)

print("Saved 1_structured_reference.csv:", final_sampled_df.shape)
print("Saved matched_final.csv:", matched_final.shape)
print("Saved 2_image_reference.csv:", image_ref.shape)
print("Saved 3_text_reference.csv:", text_ref.shape)

Saved 1_structured_reference.csv: (2200, 9)
Saved matched_final.csv: (164287, 17)
Saved 2_image_reference.csv: (3340, 4)
Saved 3_text_reference.csv: (2200, 3)
